## Recommendation System using Cosine Similarity

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
df.shape

(12294, 7)

In [4]:
df.columns

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  str    
 2   genre     12232 non-null  str    
 3   type      12269 non-null  str    
 4   episodes  12294 non-null  str    
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 1.3 MB


In [6]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [7]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["genre"] = df["genre"].fillna("Unknown")
df["type"] = df["type"].fillna(df["type"].mode()[0])
df["rating"] = df["rating"].fillna(df["rating"].mean())
#df["episodes"] = df["episodes"].replace("Unknown", 0)
#not doing above because it will convert the column to str type.
df["episodes"] = df["episodes"].replace("Unknown", np.nan)
df["episodes"] = df["episodes"].fillna(0).astype(int)

df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [10]:
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(df["genre"])

#Scaling the numeric features to bring them in range of -1 to 1.
scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(df[["rating","members","episodes"]])

#Converting sparse matrix to normal array.
combined_features = np.hstack((genre_matrix.toarray(), numeric_features))
similarity = cosine_similarity(combined_features)
print(similarity)

[[1.         0.53222643 0.46238813 ... 0.24157195 0.24807796 0.27820631]
 [0.53222643 1.         0.51705352 ... 0.20971208 0.21530718 0.24143455]
 [0.46238813 0.51705352 1.         ... 0.24117698 0.24764279 0.27771995]
 ...
 [0.24157195 0.20971208 0.24117698 ... 1.         0.99994462 0.99824864]
 [0.24807796 0.21530718 0.24764279 ... 0.99994462 1.         0.99881138]
 [0.27820631 0.24143455 0.27771995 ... 0.99824864 0.99881138 1.        ]]


In [11]:
def recommend_anime(name, n):
    if name not in df["name"].values:
        print("Anime not found.")
        return
    index = df[df["name"]==name].index[0]
    scores = list(enumerate(similarity[index])) #(index,similarity score)
    #Anonmyous function to get similarity score from list
    scores = sorted(scores, key=lambda x:x[1], reverse=True)
    scores = scores[1:n+1]    #to not recommend the same anime
    idx = [i[0] for i in scores] #list with indexes of recommended animes
    return df.loc[idx,["name","genre","rating"]]

In [12]:
recommend_anime("Naruto",5)

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32
346,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",8.16
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50


In [13]:
def recommend_threshold(name, n=0.60):
    if name not in df["name"].values:
        print("Anime not found.")
        return
    index = df[df["name"]==name].index[0]
    scores = list(enumerate(similarity[index]))
    temp = []
    for i, s in scores:
        if s >= n:
            temp.append((i, s))
            scores = temp
    scores = sorted(scores, key=lambda x:x[1], reverse=True)
    scores = scores[1:]     
    idx = [i[0] for i in scores]    
    return df.loc[idx, ["name","rating"]]

In [14]:
recommend_threshold("Naruto",0.6)

,name,rating
615,Naruto: Shippuuden,7.94
206,Dragon Ball Z,8.32
346,Dragon Ball,8.16
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,7.53
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,7.50
...,...,...
55,Tengen Toppa Gurren Lagann Movie: Lagann-hen,8.64
10551,Tiger &amp; Bunny: Too Many Cooks Spoil the Broth,6.06
1732,Yawara! Special: Zutto Kimi no Koto ga... .,7.46
2617,Munto: Toki no Kabe wo Koete,7.21


In [ ]:
#Spliting the dataset
train_df, test_df = train_test_split(df,test_size=0.2,random_state=42)

print("Training Samples:", len(train_df))
print("Testing Samples :", len(test_df))

Training Samples: 9835
Testing Samples : 2459


In [22]:
#TF-IDF Vectorization on genre
tfidf = TfidfVectorizer()

train_genre = tfidf.fit_transform(train_df["genre"])
train_genre

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 33246 stored elements and shape (9835, 48)>

In [ ]:
#Normalizing numerical features
scaler = MinMaxScaler()
train_numeric = scaler.fit_transform(train_df[["rating", "members", "episodes"]])

In [ ]:
#Combining TF-IDF and numerical features
train_features = np.hstack((train_genre.toarray(), train_numeric))
# Cosine similarity matrix
train_similarity = cosine_similarity(train_features)

print("Similarity Matrix Shape:", train_similarity.shape)

Similarity Matrix Shape: (9835, 9835)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

y_true = []
y_pred = []

for _, test_row in test_df.iterrows():
    #Transforming test genre using training vocabulary
    test_genre = tfidf.transform([test_row["genre"]])
    #Normalizing numerical values
    test_numeric = scaler.transform([[test_row["rating"],test_row["members"],test_row["episodes"]]])
    #Combining features
    test_feature = np.hstack((test_genre.toarray(),test_numeric))
    #Computing similarity with training data
    similarity_scores = cosine_similarity(test_feature,train_features)[0]
    #Finding most similar anime
    best_match = np.argmax(similarity_scores)
    predicted_genre = train_df.iloc[best_match]["genre"]
    y_true.append(test_row["genre"])
    y_pred.append(predicted_genre)

In [ ]:
precision = precision_score(y_true,y_pred,average="weighted",zero_division=0)
recall = recall_score(y_true,y_pred,average="weighted",zero_division=0)
f1 = f1_score(y_true,y_pred,average="weighted",zero_division=0)

print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F1-Score  :", round(f1, 4))

Precision : 0.7909
Recall    : 0.8174
F1-Score  : 0.8001


## Interview Questions

(Q) Can you explain the difference between user-based and item-based collaborative filtering?
* User-based filtering: Recommends items liked by users with similar preferences. Example: If User A and User B have similar movie ratings, movies liked by User B are recommended to User A.
* Item-based filtering: Recommends items similar to those the user has already liked. Example: If a user likes Movie A, the system recommends movies similar to Movie A.

(Q) What is collaborative filtering, and how does it work?
* Collaborative filtering is a recommendation technique that suggests items based on the preferences and behavior of similar users or similar items.
* It works by analyzing user-item interactions (such as ratings or purchases), finding similarities between users or items, and recommending items that the user is likely to prefer.